# relu-elementwise-max — worked example 3: Apply ReLU to a 2D Feature Map and Count Activated Units

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `relu-elementwise-max`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import matplotlib.pyplot as plt

## Concept

In convolutional networks, ReLU is applied elementwise to entire feature maps of shape `(batch, channels, height, width)`. The operation is identical to the 1D case — `max(x, 0)` per element — but the statistics of how many units activate (the "activation sparsity") become interesting to measure. High sparsity can indicate dying ReLUs or efficient feature detection.

## Worked solution

**Step 1 — create a 2D feature map.** We use a `(2, 3, 4, 4)` tensor (2 images, 3 channels, 4×4 spatial) with random values from a standard normal, which will produce roughly 50% negative values.

**Step 2 — apply ReLU via `t.maximum`.** `t.maximum(x, t.zeros_like(x))` broadcasts correctly over all dimensions. Alternatively, `t.maximum(x, t.tensor(0.0))` works since the scalar broadcasts.

**Step 3 — compute activation fraction.** Count elements where the output is positive: `(y > 0).float().mean()`. This should be close to 0.5 for zero-mean normal inputs.

**Step 4 — verify nonnegativity.** All elements of `y` must be ≥ 0. Use `(y >= 0).all()`.

**Step 5 — print summary statistics.** Mean, min, and max of input vs output illustrate the clipping effect.

In [ ]:
import torch as t

t.manual_seed(42)
batch, C, H, W = 2, 3, 4, 4
x = t.randn(batch, C, H, W)

# ReLU via elementwise max
y = t.maximum(x, t.tensor(0.0))

activation_frac = (y > 0).float().mean().item()
print(f'Input  shape: {tuple(x.shape)}')
print(f'Output shape: {tuple(y.shape)}')
print(f'Input  mean: {x.mean():.3f}, min: {x.min():.3f}, max: {x.max():.3f}')
print(f'Output mean: {y.mean():.3f}, min: {y.min():.3f}, max: {y.max():.3f}')
print(f'Activation fraction: {activation_frac:.3f} (expect ~0.5 for N(0,1))')
print(f'All outputs >= 0: {(y >= 0).all().item()}')
print(f'Negative inputs zeroed: {t.allclose(y[x < 0], t.zeros(1))}')